# 第73章 用户行为数据分析

围绕用户事件日志完成活跃、漏斗、留存和渠道对比，练习从行为明细提炼用户指标。

## 项目背景

产品团队希望知道用户从访问到支付在哪个环节流失，以及不同获客渠道带来的用户质量是否不同。每行事件包含用户、时间、渠道和行为类型。

## 学习目标

- 理解事件级数据与用户级指标
- 清洗时间和重复事件
- 计算漏斗转化和用户活跃
- 用Seaborn比较不同渠道表现


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| user_id | 用户编号 | 用户级主键 |
| event_time | 事件时间 | 用户行为发生时间 |
| event | 行为事件 | 访问、查看商品、加购、支付 |
| channel | 来源渠道 | 搜索、社交、会员 |
| page | 页面 | 落地页、商品页、购物车、收银台 |
| duration | 停留秒数 | 本次事件持续时间 |

## 数据质量检查清单

- 用户编号是否缺失
- 事件时间能否解析
- 事件名称是否在允许集合内
- 同一用户同一事件是否重复
- 停留时长是否为非负数


## 项目任务

1. 构造并检查事件日志
2. 生成用户级活跃指标
3. 按阶段计算转化漏斗
4. 比较渠道质量和行为时长
5. 提出一个可验证的产品优化方向


## 1. 准备事件日志

事件日志以用户为中心记录行为，分析前先把时间字段转成真正的时间类型。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(73)
users = [f"U{index:03d}" for index in range(1, 31)]
events = pd.DataFrame({
    "user_id": rng.choice(users, 180),
    "event_time": pd.Timestamp("2026-03-01") + pd.to_timedelta(rng.integers(0, 14 * 24 * 3600, 180), unit="s"),
    "event": rng.choice(["访问", "查看商品", "加入购物车", "支付"], 180, p=[0.34, 0.30, 0.22, 0.14]),
    "channel": rng.choice(["搜索", "社交", "会员"], 180, p=[0.45, 0.35, 0.20]),
    "page": rng.choice(["落地页", "商品页", "购物车", "收银台"], 180),
    "duration": rng.integers(5, 420, 180),
})
events["event_time"] = pd.to_datetime(events["event_time"])
print(events.head())
print("用户数:", events["user_id"].nunique())


## 2. 做事件质量检查

质量检查既要统计问题规模，也要明确后续处理方式；重复事件不能直接当作真实转化。


In [ ]:
raw_events = events.copy()
raw_events.loc[0, "duration"] = -4
raw_events = pd.concat([raw_events, raw_events.iloc[[1]]], ignore_index=True)

allowed_events = {"访问", "查看商品", "加入购物车", "支付"}
print("重复行:", raw_events.duplicated().sum())
print("缺失值:\n", raw_events.isna().sum())
print("非法事件:", (~raw_events["event"].isin(allowed_events)).sum())
print("负停留时长:", (raw_events["duration"] < 0).sum())

clean_events = raw_events.drop_duplicates().copy()
clean_events["duration"] = clean_events["duration"].clip(lower=0)
clean_events = clean_events[clean_events["event"].isin(allowed_events)].copy()
clean_events["date"] = clean_events["event_time"].dt.date
print("清洗后事件数:", len(clean_events))


## 3. 计算活跃和漏斗指标

漏斗按用户去重，而不是按事件行数计数；同一用户重复访问仍只算一个进入用户。


In [ ]:
user_metrics = clean_events.groupby("user_id").agg(
    active_days=("date", "nunique"),
    event_count=("event", "size"),
    average_duration=("duration", "mean"),
    channel=("channel", "first"),
).reset_index()
user_metrics["paid"] = user_metrics["user_id"].isin(clean_events.loc[clean_events["event"] == "支付", "user_id"])

funnel_order = ["访问", "查看商品", "加入购物车", "支付"]
funnel = clean_events.groupby("event")["user_id"].nunique().reindex(funnel_order, fill_value=0).rename("users").reset_index()
funnel["conversion_from_visit"] = funnel["users"] / funnel.loc[0, "users"]
funnel["step_conversion"] = funnel["users"] / funnel["users"].shift(1)
print("用户指标摘要:\n", user_metrics.describe(numeric_only=True).round(2))
print("\n漏斗:\n", funnel.round(3))


## 4. 比较渠道质量

渠道比较同时查看用户规模、支付率和平均停留，避免只根据用户数量评价渠道。


In [ ]:
channel_summary = user_metrics.groupby("channel").agg(
    users=("user_id", "nunique"),
    paid_users=("paid", "sum"),
    average_duration=("average_duration", "mean"),
    average_active_days=("active_days", "mean"),
).reset_index()
channel_summary["payment_rate"] = channel_summary["paid_users"] / channel_summary["users"]
print(channel_summary.round(3))

sns.set_theme(style="whitegrid", context="notebook")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(data=channel_summary, x="channel", y="payment_rate", hue="channel", legend=False, ax=axes[0], color="#1a73e8")
axes[0].set(title="渠道支付率", xlabel="渠道", ylabel="支付率")
sns.barplot(data=channel_summary, x="channel", y="average_duration", hue="channel", legend=False, ax=axes[1], color="#188038")
axes[1].set(title="渠道平均停留", xlabel="渠道", ylabel="秒")
fig.tight_layout()
plt.show()


## 5. 输出产品建议

建议要指向一个具体漏斗环节，并说明下一步需要验证的指标。


In [ ]:
largest_loss = funnel.iloc[1:]["step_conversion"].idxmin()
loss_event = funnel.loc[largest_loss, "event"]
best_channel = channel_summary.loc[channel_summary["payment_rate"].idxmax(), "channel"]
best_rate = channel_summary["payment_rate"].max()
print(f"最大阶段损失出现在进入“{loss_event}”之前，建议检查上一阶段页面体验。")
print(f"当前支付率最高的渠道是“{best_channel}”，支付率为 {best_rate:.1%}。")
print("下一步：按设备、页面版本和日期拆分漏斗，验证问题是否集中在特定人群。")


## 结论与表达

- 事件级明细需要先转换成用户级指标
- 漏斗每一步的分母要明确
- 渠道质量应同时观察规模、转化和行为深度


## 项目验收清单

- 从头运行不报错
- 包含事件字段质量检查
- 给出用户级活跃指标
- 漏斗至少有两种转化率
- 建议能对应一个具体数据指标

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

围绕用户事件日志完成活跃、漏斗、留存和渠道对比，练习从行为明细提炼用户指标。


### 你已经完成

- 理解事件级数据与用户级指标
- 清洗时间和重复事件
- 计算漏斗转化和用户活跃
- 用Seaborn比较不同渠道表现


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 构造并检查事件日志 |
| 步骤 2 | 生成用户级活跃指标 |
| 步骤 3 | 按阶段计算转化漏斗 |
| 步骤 4 | 比较渠道质量和行为时长 |
| 步骤 5 | 提出一个可验证的产品优化方向 |


### 质量与结论提醒

- 用户编号是否缺失
- 事件时间能否解析
- 事件名称是否在允许集合内
- 事件级明细需要先转换成用户级指标
- 漏斗每一步的分母要明确
- 渠道质量应同时观察规模、转化和行为深度


### 项目交付检查

- [ ] 从头运行不报错
- [ ] 包含事件字段质量检查
- [ ] 给出用户级活跃指标
- [ ] 漏斗至少有两种转化率
- [ ] 建议能对应一个具体数据指标
